# Spec-FastGS — Mip-NeRF 360 Ablation Sweep & Upload Pipeline

> **Kernel**: `thesis_env` (Set via Kernel → Change Kernel after running `bosch_setup_thesis.ipynb` once to register it)

Runs the Spec-FastGS training sweep on Mip-NeRF 360 scenes inside the BOSCH server environment across a configurable list of `(resolution, ablation-config)` runs, then pushes each run's **entire output folder** (no zip) straight to its own Hugging Face dataset repo.

**What this notebook does:**
1. **Proxy & Env Check**: Sets up environment variables for the BOSCH server.
2. **Runs Definition**: A `RUNS` list of `{resolution, config_label, flags, hf_repo}` — each entry is one full sweep + upload.
3. **Resolution & Layout Validation**: Verifies the resolution folders used by `RUNS` exist for all 9 scenes (`bicycle`, `flowers`, `garden`, `stump`, `treehill`, `room`, `counter`, `kitchen`, `bonsai`).
4. **Sweep Execution**: For each run, exports `IMAGES` + the 5 ablation flags (`EXTRACT_REF_PRIOR`, `BACKUP_REF_PRIOR`, `USE_REF_SCORE`, `USE_ADAPTIVE_PRIOR`, `USE_SH_SPEC_MASK`) and invokes `run_mip360.sh`.
5. **Results Summary**: Formats and prints quantitative metrics (`results_grouped.json`) in a neat table for each run.
6. **Whole-Folder HF Upload**: Uploads `output/mip360_{resolution}` directly (via `HfApi.upload_folder`, no zip) to that run's dedicated dataset repo.
7. **Auto Cleanup**: Purges the local output folder after a successful upload to save BOSCH server disk space.

## c00 — Proxy Settings
Sets the BOSCH proxy for external connectivity.

In [ ]:
# ── Proxy (required for HF / huggingface cache / diagnostic endpoints) ────────
import os

PROXY = 'http://rb-proxy-sl.bosch.com:8080'
HOME  = os.path.expanduser('~')

os.environ['http_proxy']  = PROXY
os.environ['https_proxy'] = PROXY
os.environ['HTTP_PROXY']  = PROXY
os.environ['HTTPS_PROXY'] = PROXY

print(f'Proxy set to: {PROXY}')

## c01 — Config, Target Resolutions & Kernel Check
Defines paths, target resolution list (`["images", "images_2", "images_4", "images_8"]`), and double-checks if the correct virtual environment kernel is loaded.

In [ ]:
# ── Configurations & environment variables check ─────────────────────────────
import os
import sys

HOME = os.path.expanduser('~')

# Robustly resolve repository root path
if os.path.isdir('/home/ghp4hc/thesis-all/spec-fastgs'):
    REPO_ROOT = '/home/ghp4hc/thesis-all/spec-fastgs'
elif os.path.isdir(os.path.join(os.getcwd(), 'thesis-all', 'spec-fastgs')):
    REPO_ROOT = os.path.join(os.getcwd(), 'thesis-all', 'spec-fastgs')
else:
    REPO_ROOT = os.path.join(os.getcwd(), 'spec-fastgs')

ENV_NAME = 'thesis_env'

ALL_TRUE_FLAGS = {
    'EXTRACT_REF_PRIOR': 'True',
    'BACKUP_REF_PRIOR': 'True',
    'USE_REF_SCORE': 'True',
    'USE_ADAPTIVE_PRIOR': 'True',
    'USE_SH_SPEC_MASK': 'True',
}
ALL_FALSE_FLAGS = {
    'EXTRACT_REF_PRIOR': 'False',
    'BACKUP_REF_PRIOR': 'False',
    'USE_REF_SCORE': 'False',
    'USE_ADAPTIVE_PRIOR': 'False',
    'USE_SH_SPEC_MASK': 'False',
}

# Each run = one full mip360 sweep at a given resolution + ablation config,
# uploaded (whole folder, no zip) to its own dedicated HF dataset repo.
RUNS = [
    {'resolution': 'images_4', 'config_label': 'False', 'flags': ALL_FALSE_FLAGS,
     'hf_repo': 'DiBiay/spec-fastgs-False-mipneft360-images4'},
    {'resolution': 'images_2', 'config_label': 'True', 'flags': ALL_TRUE_FLAGS,
     'hf_repo': 'DiBiay/spec-fastgs-True-mipneft360-images2'},
    {'resolution': 'images_2', 'config_label': 'False', 'flags': ALL_FALSE_FLAGS,
     'hf_repo': 'DiBiay/spec-fastgs-False-mipneft360-images2'},
]

# Resolutions actually needed on disk, derived from RUNS (for the layout check below)
TARGET_RESOLUTIONS = sorted({r['resolution'] for r in RUNS})

print(f'Active Python        : {sys.executable}')
print(f'Active Kernel name   : {ENV_NAME}')
print(f'Repository Root      : {REPO_ROOT}')
print(f'Target Resolutions   : {TARGET_RESOLUTIONS}')
print(f'Planned runs:')
for r in RUNS:
    print(f"  - resolution={r['resolution']:<10s} config={r['config_label']:<5s} -> {r['hf_repo']}")

assert REPO_ROOT in sys.executable or ENV_NAME in sys.executable or '.conda' in sys.executable, \
    f"WARNING: You are not running on the '{ENV_NAME}' kernel! Please select Kernel -> Change Kernel -> Python ({ENV_NAME})"

## c02 — Imports & GPU Validation
Verifies hardware detection and compiled custom modules availability.

In [ ]:
# ── Verification of PyTorch & custom submodules ────────────────────────────────
import torch
print('PyTorch version :', torch.__version__)
print('CUDA Available  :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU device name :', torch.cuda.get_device_name(0))
    print('Compute Cap.    :', torch.cuda.get_device_capability(0))

import diff_gaussian_rasterization_fastgs
import simple_knn
import fused_ssim
print('rasterizer      : OK')
print('simple-knn      : OK')
print('fused-ssim      : OK')

try:
    import huggingface_hub
    print('huggingface_hub : OK')
except ImportError:
    print('huggingface_hub : MISSING (will auto-install during the upload step)')

## c03 — Verify Dataset Path
Checks that the Mip-NeRF 360 source dataset is available on the server.

In [ ]:
# ── Verify Dataset Directory ──────────────────────────────────────────────────
import os
import sys

src_root = "/home/ghp4hc/datasets/datasets/mipneft360"
assert os.path.exists(src_root), f"Dataset path not found at {src_root}! Check that the datasets are downloaded."
print(f"✅ Found dataset source root: {src_root}")

print(f"\n📂 Source datasets directory content:")
print(os.listdir(src_root))

## c04 — Verify Target Resolutions Layout
Validates all 9 scenes for each resolution in `TARGET_RESOLUTIONS`.

In [ ]:
# ── Verify Mip-NeRF 360 Dataset Layout for all Target Resolutions ────────────
import os

src_root = "/home/ghp4hc/datasets/datasets/mipneft360"
MIP360_SCENES = [
    "bicycle", "flowers", "garden", "stump", "treehill",
    "room", "counter", "kitchen", "bonsai",
]

for resolution in TARGET_RESOLUTIONS:
    print(f"\n🔍 Verifying resolution '{resolution}' under {src_root} ...")
    missing = []
    for scene in MIP360_SCENES:
        v2_path = os.path.join(src_root, "360_v2", scene)
        extra_path = os.path.join(src_root, "360_extra_scenes", scene)
        
        if os.path.isdir(v2_path):
            scene_dir = v2_path
        elif os.path.isdir(extra_path):
            scene_dir = extra_path
        else:
            scene_dir = None
            
        if scene_dir is None:
            status = "MISSING (scene folder not found)"
            missing.append(scene)
        else:
            images_dir = os.path.join(scene_dir, resolution)
            if not os.path.isdir(images_dir):
                status = f"MISSING ({resolution} not found; has: {sorted(os.listdir(scene_dir))[:6]})"
                missing.append(scene)
            else:
                n_imgs = len([f for f in os.listdir(images_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
                status = f"OK ({n_imgs} images)"
                
        print(f"  {scene:<12s} {status}")

    if missing:
        print(f"⚠️  {len(missing)}/{len(MIP360_SCENES)} scene(s) missing {resolution}: {missing}")
    else:
        print(f"✅ All {len(MIP360_SCENES)} scenes verified for '{resolution}'.")

## c05 — Run Spec-FastGS Sweeps, Upload & Auto-Cleanup
Iterates through each entry in `RUNS`, runs `run_mip360.sh` with that run's resolution + ablation flags, summarizes metrics, uploads the whole output folder (no zip) to that run's dedicated HF dataset repo, and purges local output to save disk space.

In [ ]:
# ── Prepare script inputs & HF API ───────────────────────────────────────────
import subprocess
import os
import sys
import json
import shutil

# Ensure BOSCH Proxy environment variables & explicit proxy dictionary are set
PROXY = 'http://rb-proxy-sl.bosch.com:8080'
os.environ['http_proxy']  = PROXY
os.environ['https_proxy'] = PROXY
os.environ['HTTP_PROXY']  = PROXY
os.environ['HTTPS_PROXY'] = PROXY

PROXIES_DICT = {
    'http': PROXY,
    'https': PROXY,
}

try:
    import huggingface_hub
    from huggingface_hub import HfApi
except ImportError:
    print("Installing huggingface_hub via pip...")
    subprocess.run([sys.executable, "-m", "pip", "install", "huggingface_hub", "--proxy", PROXY], check=True)
    import huggingface_hub
    from huggingface_hub import HfApi

# Configure global HTTP backend for huggingface_hub with BOSCH proxy
def proxy_session_factory():
    import requests
    session = requests.Session()
    session.proxies = PROXIES_DICT
    return session

try:
    huggingface_hub.configure_http_backend(backend_factory=proxy_session_factory)
except Exception:
    pass

DATA_ROOT = "/home/ghp4hc/datasets/datasets/mipneft360"
HF_TOKEN = os.environ.get('HF_TOKEN', '').strip()
if not HF_TOKEN:
    HF_TOKEN = input("Enter your Hugging Face Access Token (WRITE permission required): ").strip()

venv_bin = os.path.dirname(sys.executable)
cuda_home = os.environ.get('CUDA_HOME', '')
if not cuda_home:
    search_script = 'which nvcc 2>/dev/null || (for init in /etc/profile /etc/profile.d/modules.sh; do [ -f "$init" ] && source "$init"; done && for mod in cuda/11.7 cuda/11.8 cuda/12.1 cuda/12.6; do module load "$mod" 2>/dev/null; done && which nvcc 2>/dev/null)'
    r_nvcc = subprocess.run(['bash', '-c', search_script], capture_output=True, text=True)
    if r_nvcc.returncode == 0 and r_nvcc.stdout.strip():
        cuda_home = os.path.dirname(os.path.dirname(r_nvcc.stdout.strip()))
    else:
        cuda_home = '/usr/local/cuda'

MIP360_SCENES = [
    "bicycle", "flowers", "garden", "stump", "treehill",
    "room", "counter", "kitchen", "bonsai",
]

def fmt(x, nd=4):
    return f"{x:.{nd}f}" if isinstance(x, (int, float)) else "-"

for run in RUNS:
    resolution = run['resolution']
    config_label = run['config_label']
    flags = run['flags']
    hf_repo = run['hf_repo']

    print(f"\n========================================================================")
    print(f" 🚀 STARTING SPEC-FASTGS SWEEP: resolution={resolution}  config={config_label}")
    print(f"========================================================================")

    logfile = os.path.join(os.path.dirname(REPO_ROOT), f"mip360_{resolution}_{config_label}_specfastgs_run.log")

    # 1. Run bash sweep with this run's resolution + ablation flags
    flag_exports = "\n    ".join(f"export {k}={v}" for k, v in flags.items())
    cmd = f'''
    export PATH={venv_bin}:{cuda_home}/bin:$PATH
    export LD_LIBRARY_PATH={cuda_home}/lib64:$LD_LIBRARY_PATH
    export CUDA_VISIBLE_DEVICES=0
    export DATA_ROOT={DATA_ROOT}
    export IMAGES={resolution}
    {flag_exports}
    cd "{REPO_ROOT}"
    bash run_mip360.sh > "{logfile}" 2>&1
    '''
    r = subprocess.run(['bash', '-c', cmd])

    print(f"--- tail of {logfile} ---")
    if os.path.exists(logfile):
        with open(logfile, 'r') as f:
            lines = f.readlines()
            print(''.join(lines[-40:]))

    # 2. Print quantitative summary
    out_root = os.path.join(REPO_ROOT, "output", f"mip360_{resolution}")
    print(f"\n📊 Quantitative Results Summary: resolution={resolution} config={config_label}")
    header = f"{'scene':<12s}{'PSNR':>8s}{'SSIM':>8s}{'LPIPS':>8s}{'Spec_PSNR':>11s}{'ASG_IoU':>9s}{'#Gauss':>10s}{'time':>10s}"
    print(header)
    print("-" * len(header))
    for scene in MIP360_SCENES:
        out_dir = os.path.join(out_root, scene)
        results_path = os.path.join(out_dir, "results_grouped.json")
        info_path = os.path.join(out_dir, "train_info.json")
        if not os.path.exists(results_path):
            print(f"{scene:<12s}  (no results_grouped.json)")
            continue
        with open(results_path) as f:
            res = json.load(f)
        sr = next(iter(res.values()))
        rr = next(iter(sr.values()))
        main = rr.get("main_metrics", {})
        aux = rr.get("aux_metrics", {})
        info = {}
        if os.path.exists(info_path):
            with open(info_path) as f:
                info = json.load(f)
        print(f"{scene:<12s}{fmt(main.get('PSNR')):>8s}{fmt(main.get('SSIM')):>8s}{fmt(main.get('LPIPS')):>8s}"
              f"{fmt(aux.get('Spec_PSNR')):>11s}{fmt(aux.get('ASG_Residual_IoU')):>9s}"
              f"{str(info.get('final_gaussians', '-')):>10s}{str(info.get('training_time_formatted', '-')):>10s}")

    # 3. Push the whole output folder to Hugging Face (no zip)
    if os.path.isdir(out_root):
        # Strip point_cloud folders (large, unused after rendering) before uploading
        for scene in MIP360_SCENES:
            pc_dir = os.path.join(out_root, scene, "point_cloud")
            if os.path.isdir(pc_dir):
                shutil.rmtree(pc_dir, ignore_errors=True)

        print(f"\n📤 Uploading whole folder '{out_root}' -> Hugging Face dataset '{hf_repo}'...")
        try:
            api = HfApi(proxies=PROXIES_DICT)
            try:
                api.repo_info(repo_id=hf_repo, repo_type="dataset", token=HF_TOKEN)
            except Exception as repo_err:
                if "404" in str(repo_err) or "Repository Not Found" in str(repo_err):
                    print(f"➕ Creating dataset repository '{hf_repo}'...")
                    api.create_repo(repo_id=hf_repo, repo_type="dataset", token=HF_TOKEN, private=True)
                else:
                    print(f"ℹ️ Repo info status note: {repo_err}")

            api.upload_folder(
                folder_path=out_root,
                repo_id=hf_repo,
                repo_type="dataset",
                token=HF_TOKEN,
            )
            print(f"🎉 [SUCCESS] Uploaded '{out_root}' to '{hf_repo}'!")

            # 4. Purge local output directory to save disk space
            print(f"🗑️ Cleaning up output directory '{out_root}'...")
            shutil.rmtree(out_root, ignore_errors=True)
            print(f"✅ Local disk space freed for resolution '{resolution}' config '{config_label}'!")
        except Exception as e:
            print(f"❌ [ERROR] HF upload failed for {resolution}/{config_label}: {e}")
            print(f"⚠️  Retaining output folder '{out_root}' for inspection.")
    else:
        print(f"❌ [ERROR] Output directory '{out_root}' not found. Skipping upload for {resolution}/{config_label}.")